In [50]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Linux


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = '/media/cat/4TB/donato/andres/DON011733/20230118/calibration/Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)


print ("DONE...")

memmap :  (20000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (93, 20000)
         self.Fneu (neuropile):  (93, 20000)
         self.iscell (Suite2p cell classifier output):  (306, 2)
              of which number of good cells:  (93,)
         self.spks (deconnvoved spikes):  (93, 20000)
         self.stat (footprints structure):  (93,)
         mean std over all cells :  57.0
# of footprints;  93
DONE...


In [60]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

#
bmi_c.scale=1.5
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

# visualize traces
bmi_c.vmin=0
bmi_c.vmax=5000
bmi_c.max_n_cells = 50
bmi_c.visualize_traces_snr_order(bmi_c.std_map)

# 


computing roi traces for SNR indexing: 100%|█████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1564.74it/s]


memmap :  (20000, 512, 512)


In [61]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [0,1]
bmi_c.ensemble2 = [10,12]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles2(bmi_c.std_map)

print ("DONE...")


all cells: [ 0  1 10 12]


100%|█████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:00<00:00, 27553.40it/s]


cell ids:  [ 0  1 10 12]
DONE...


In [62]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
bmi_c.find_reward_thresholds_high_realtime()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  666 max # of random rewards (i.e. every 30sec)  22
 @30% reward:  5
 high guess:  2.0285086596619175
updated rewards #:  1 2.0285086596619175
updated rewards #:  1 1.9270832266788216
updated rewards #:  1 1.8307290653448804
updated rewards #:  1 1.7391926120776364
updated rewards #:  2 1.6522329814737546
updated rewards #:  3 1.5696213324000667
updated rewards #:  5 1.4911402657800634
updated rewards #:  5 1.4911402657800634
thresholds:  1.4911402657800634
 PROCESSING...
bmi_c.high: scaled by:  1.0 , final value:   1.4911402657800634


In [63]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
 couldn't save bmi_c.object .... TO FIX!
Done...
